In [11]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import pandas as pd
from backtesting.test import SMA, GOOG

In [12]:
#读取文件
bu = pd.read_csv('data/bu.csv')
jd = pd.read_csv('data/jd.csv')
l = pd.read_csv('data/l.csv')
pp = pd.read_csv('data/pp.csv')
ru = pd.read_csv('data/ru.csv')
v = pd.read_csv('data/v.csv')
bu

,date,open,high,low,close,volume,hold,settle
0,2015-06-26,2948.0,2972.0,2936.0,2960.0,32378,50834,2954.0
1,2015-06-29,2948.0,2970.0,2922.0,2936.0,31124,49566,2952.0
2,2015-06-30,2930.0,2948.0,2804.0,2812.0,52186,48312,2854.0
3,2015-07-01,2832.0,2850.0,2808.0,2826.0,31576,49450,2834.0
4,2015-07-02,2826.0,2830.0,2774.0,2816.0,36706,50466,2802.0
...,...,...,...,...,...,...,...,...
2427,2025-06-20,3738.0,3789.0,3723.0,3747.0,253286,284513,3751.0
2428,2025-06-23,3736.0,3804.0,3719.0,3781.0,317501,311332,3769.0
2429,2025-06-24,3762.0,3770.0,3562.0,3580.0,460577,269578,3643.0
2430,2025-06-25,3565.0,3583.0,3537.0,3574.0,282213,251579,3556.0


In [13]:
#重命名各列
def column_rename(df):
    df=df.rename(columns={
        df.columns[0]: 'Date',
        df.columns[1]: 'Open',
        df.columns[2]: 'High',
        df.columns[3]: 'Low',
        df.columns[4]: 'Close',
        df.columns[5]: 'Volume'
    }).drop(columns=[df.columns[6], df.columns[7]])
    df['Date'] = pd.to_datetime(df['Date'])
    return df.set_index('Date')
    
new_bu = column_rename(bu)

new_jd = column_rename(jd)
new_l = column_rename(l)
new_pp = column_rename(pp)
new_ru = column_rename(ru)
new_v = column_rename(v)
print(new_jd.head())
GOOG.head()

              Open    High     Low   Close  Volume
Date                                              
2015-06-26  4145.0  4145.0  4073.0  4085.0   76016
2015-06-29  4065.0  4090.0  4045.0  4068.0   70708
2015-06-30  4060.0  4060.0  3965.0  3969.0   96574
2015-07-01  4000.0  4112.0  3992.0  4028.0   69972
2015-07-02  4026.0  4029.0  3983.0  4009.0   54432


,Open,High,Low,Close,Volume
2004-08-19,100.00,104.06,95.96,100.34,22351900
2004-08-20,101.01,109.08,100.50,108.31,11428600
2004-08-23,110.75,113.48,109.05,109.40,9137200
2004-08-24,111.24,111.60,103.57,104.87,7631300
2004-08-25,104.96,108.00,103.88,106.00,4598900


In [26]:
# 纯双均值策略（0）
class SmaCross(Strategy):
    def init(self):
        price = self.data.Close
        self.ma1 = self.I(SMA, price, 10)
        self.ma2 = self.I(SMA, price, 20)

    def next(self):
        if crossover(self.ma1, self.ma2):
            self.buy()
        elif crossover(self.ma2, self.ma1):
            self.sell()

In [15]:
#ATR计算函数
def ATR(df, period=14):
    high_low = df['High'] - df['Low']
    high_close = abs(df['High'] - df['Close'].shift())
    low_close = abs(df['Low'] - df['Close'].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()


In [16]:
# 止损卖出（2）
class SmaCross(Strategy):
    atr_period = 14
    atr_multiplier = 1.5
    sma_filter = 50
    
    def init(self):
        # 基础均线
        price = self.data.Close
        self.ma1 = self.I(SMA, price, 10)
        self.ma2 = self.I(SMA, price, 20)
        
        # 波动性指标
        self.atr = self.I(ATR, self.data.df, self.atr_period)
        
        # 趋势过滤指标
        self.filter = self.I(SMA, price, self.sma_filter)
        
        self.entry_price = 0
        self.stop_loss = 0

    def next(self):
        current_close = self.data.Close[-1]
        
        # 修改买入条件：添加持仓状态检查
        if self.entry_price ==0:
            if crossover(self.ma1, self.ma2) :
                self.buy()
                self.entry_price = current_close
                self.stop_loss = current_close - self.atr[-1] * self.atr_multiplier
        
        # 强化卖出条件：添加持仓状态检查
        elif self.entry_price != 0:
            # 动态更新止损位
            self.stop_loss = min(self.stop_loss, current_close - self.atr[-1] * self.atr_multiplier)
            
            if crossover(self.ma2, self.ma1) or current_close < self.stop_loss:
                self.sell()
                self.entry_price = 0  # 重置入场价格
                self.stop_loss = 0    # 重置止损位

In [17]:
# 尝试提前操作（3）
class SmaCross(Strategy):
    atr_period = 14
    atr_multiplier = 1.5
    sma_filter = 50
    
    def init(self):
        # 基础均线
        price = self.data.Close
        self.ma0 = self.I(SMA, price, 1)
        self.ma1 = self.I(SMA, price, 10)
        self.ma2 = self.I(SMA, price, 20)
        # 波动性指标
        self.atr = self.I(ATR, self.data.df, self.atr_period)
        # 趋势过滤指标
        self.filter = self.I(SMA, price, self.sma_filter)
        self.entry_price = 0
        self.stop_loss = 0
        self.current_date = self.data.df.index[0]
        self.cross1 = self.current_date
        self.cross2 = self.current_date
        self.fall1 = self.current_date
        self.fall2 = self.current_date

    def next(self):
        current_close = self.data.Close[-1]
        self.current_date = self.data.df.index[-1]
        self.cross1 = self.current_date if crossover(self.ma0, self.ma1) else self.cross1
        self.cross2 = self.current_date if crossover(self.ma0, self.ma2) else self.cross2
        self.fall1 = self.current_date if crossover(self.ma1, self.ma0) else self.fall1
        self.fall2 = self.current_date if crossover(self.ma2, self.ma0) else self.fall2
        below_ma50 = self.data.Close[-1] < self.filter[-1]
        if self.entry_price ==0:
            buy = False
            if self.cross2==self.current_date and below_ma50 and max(self.cross1,self.fall1,self.fall2)==self.cross1 and below_ma50:
                buy = True
            if buy:
                self.buy()
                self.entry_price = current_close  # 记录入场价格
                self.stop_loss = current_close - self.atr[-1] * self.atr_multiplier  # 初始止损位
        
        elif self.entry_price != 0:
            sell = False
            self.stop_loss = min(self.stop_loss, current_close - self.atr[-1] * self.atr_multiplier)
            if 0.95 * self.entry_price >= current_close:
                sell = True
            if current_close < self.stop_loss :
                sell = True
            elif self.fall2==self.current_date and max(self.cross1,self.fall1,self.cross2)==self.fall1 and not below_ma50:
                sell = True
            if sell:
                self.sell()
                self.entry_price = 0  # 重置入场价格
                self.stop_loss = 0  # 重置止损位


In [18]:
# 避免高位买入（4）
class SmaCross(Strategy):
    atr_period = 14
    atr_multiplier = 1.5
    sma_filter = 50
    
    def init(self):
        # 基础均线
        price = self.data.Close
        self.ma0 = self.I(SMA, price, 1)
        self.ma1 = self.I(SMA, price, 10)
        self.ma2 = self.I(SMA, price, 20)
        # 波动性指标
        self.atr = self.I(ATR, self.data.df, self.atr_period)
        # 趋势过滤指标
        self.filter = self.I(SMA, price, self.sma_filter)
        self.entry_price = 0
        self.sell_price = price* 1.5
        self.stop_loss = 0
        self.current_date = self.data.df.index[0]
        self.cross1 = self.current_date
        self.cross2 = self.current_date
        self.fall1 = self.current_date
        self.fall2 = self.current_date

    def next(self):
        current_close = self.data.Close[-1]
        self.current_date = self.data.df.index[-1]
        self.cross1 = self.current_date if crossover(self.ma0, self.ma1) else self.cross1
        self.cross2 = self.current_date if crossover(self.ma0, self.ma2) else self.cross2
        self.fall1 = self.current_date if crossover(self.ma1, self.ma0) else self.fall1
        self.fall2 = self.current_date if crossover(self.ma2, self.ma0) else self.fall2
        below_ma50 = self.data.Close[-1] < self.filter[-1]
        if self.entry_price ==0:
            buy = False
            self.sell_price = 1.01 * self.sell_price
            if self.cross2==self.current_date and below_ma50 and max(self.cross1,self.fall1,self.fall2)==self.cross1 and below_ma50:
                if self.sell_price != 0 and self.sell_price > current_close:
                    buy = True
            if buy:
                self.buy()
                self.entry_price = current_close  # 记录入场价格
                self.stop_loss = current_close - self.atr[-1] * self.atr_multiplier  # 初始止损位
                self.sell_price = 0  # 重置卖出价格
        
        elif self.entry_price != 0:
            sell = False
            self.stop_loss = min(self.stop_loss, current_close - self.atr[-1] * self.atr_multiplier)
            if 0.95 * self.entry_price >= current_close:
                sell = True
            if current_close < self.stop_loss :
                sell = True
            elif self.fall2==self.current_date and max(self.cross1,self.fall1,self.cross2)==self.fall1 and not below_ma50:
                sell = True
            if sell:
                self.sell()
                self.entry_price = 0  # 重置入场价格
                self.stop_loss = 0  # 重置止损位
                self.sell_price = current_close  # 记录卖出价格


In [19]:
# 尝试抓住迅猛上涨的行情（5）
class SmaCross(Strategy):
    atr_period = 14
    atr_multiplier = 1.5
    sma_filter = 50
    
    def init(self):
        # 基础均线
        price = self.data.Close
        self.ma0 = self.I(SMA, price, 1)
        self.ma1 = self.I(SMA, price, 10)
        self.ma2 = self.I(SMA, price, 20)
        # 波动性指标
        self.atr = self.I(ATR, self.data.df, self.atr_period)
        # 趋势过滤指标
        self.filter = self.I(SMA, price, self.sma_filter)
        self.entry_price = 0
        self.sell_price = price* 1.5
        self.stop_loss = 0
        self.current_date = self.data.df.index[0]
        self.cross1 = self.current_date
        self.cross2 = self.current_date
        self.fall1 = self.current_date
        self.fall2 = self.current_date

    def next(self):
        current_close = self.data.Close[-1]
        self.current_date = self.data.df.index[-1]
        self.cross1 = self.current_date if crossover(self.ma0, self.ma1) else self.cross1
        self.cross2 = self.current_date if crossover(self.ma0, self.ma2) else self.cross2
        self.fall1 = self.current_date if crossover(self.ma1, self.ma0) else self.fall1
        self.fall2 = self.current_date if crossover(self.ma2, self.ma0) else self.fall2
        below_ma50 = self.data.Close[-1] < self.filter[-1]
        if self.entry_price ==0:
            buy = False
            self.sell_price = 1.01 * self.sell_price
            if self.cross2==self.current_date and max(self.cross1,self.fall1,self.fall2)==self.cross1 and self.sell_price > current_close:
                if below_ma50:
                    buy = True
            if self.ma0[-1] > self.ma1[-1] and self.ma1[-1] > self.ma2[-1] and self.ma2[-1] > self.filter[-1]:
                buy = True
            if buy:
                self.buy()
                self.entry_price = current_close  # 记录入场价格
                self.stop_loss = current_close - self.atr[-1] * self.atr_multiplier  # 初始止损位
                self.sell_price = 0  # 重置卖出价格
        
        elif self.entry_price != 0:
            sell = False
            self.stop_loss = min(self.stop_loss, current_close - self.atr[-1] * self.atr_multiplier)
            if 0.95 * self.entry_price >= current_close:
                sell = True
            if current_close < self.stop_loss :
                sell = True
            elif self.fall2==self.current_date and max(self.cross1,self.fall1,self.cross2)==self.fall1 and not below_ma50:
                sell = True
            if sell:
                self.sell()
                self.entry_price = 0  # 重置入场价格
                self.stop_loss = 0  # 重置止损位
                self.sell_price = current_close  # 记录卖出价格


In [27]:
bt = Backtest(new_bu, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True)
stats = bt.run()
bt.plot()
stats

Backtest.run:   0%|          | 0/2412 [00:00<?, ?bar/s]

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                    98.64309
Equity Final [$]                    34499.202
Equity Peak [$]                    108151.156
Commissions [$]                     32485.368
Return [%]                           -65.5008
Buy & Hold Return [%]                36.30451
Return (Ann.) [%]                   -10.44115
Volatility (Ann.) [%]                26.82602
CAGR [%]                             -7.07854
Sharpe Ratio                         -0.38922
Sortino Ratio                        -0.50831
Calmar Ratio                         -0.15332
Alpha [%]                           -68.33021
Beta                                  0.07794
Max. Drawdown [%]                   -68.10094
Avg. Drawdown [%]                   -25.02693
Max. Drawdown Duration     3593 days 00:00:00
Avg. Drawdown Duration     1205 days 00:00:00
# Trades                          

In [28]:
bt = Backtest(new_jd, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True)
stats = bt.run()
bt.plot()
stats

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                     97.9852
Equity Final [$]                    16150.916
Equity Peak [$]                    116938.624
Commissions [$]                       23040.5
Return [%]                          -83.84908
Buy & Hold Return [%]               -18.65261
Return (Ann.) [%]                   -17.21443
Volatility (Ann.) [%]                28.64592
CAGR [%]                             -11.8184
Sharpe Ratio                         -0.60094
Sortino Ratio                        -0.69177
Calmar Ratio                         -0.19925
Alpha [%]                           -88.05119
Beta                                 -0.22528
Max. Drawdown [%]                   -86.39721
Avg. Drawdown [%]                    -8.36388
Max. Drawdown Duration     3549 days 00:00:00
Avg. Drawdown Duration      327 days 00:00:00
# Trades                          

In [29]:
bt = Backtest(new_l, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True)
stats = bt.run()
bt.plot()
stats

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                    97.53188
Equity Final [$]                    73113.908
Equity Peak [$]                     133374.96
Commissions [$]                     50425.892
Return [%]                          -26.88609
Buy & Hold Return [%]               -22.05019
Return (Ann.) [%]                    -3.19404
Volatility (Ann.) [%]                17.44269
CAGR [%]                             -2.13709
Sharpe Ratio                         -0.18312
Sortino Ratio                        -0.25698
Calmar Ratio                         -0.07011
Alpha [%]                           -27.07596
Beta                                 -0.00861
Max. Drawdown [%]                   -45.55657
Avg. Drawdown [%]                    -6.50444
Max. Drawdown Duration     1708 days 00:00:00
Avg. Drawdown Duration      199 days 00:00:00
# Trades                          

In [30]:
bt = Backtest(new_pp, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True)
stats = bt.run()
bt.plot()
stats

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                     98.0255
Equity Final [$]                    42142.462
Equity Peak [$]                    139601.612
Commissions [$]                     42049.358
Return [%]                          -57.85754
Buy & Hold Return [%]                -13.4332
Return (Ann.) [%]                    -8.56803
Volatility (Ann.) [%]                17.92077
CAGR [%]                             -5.78685
Sharpe Ratio                         -0.47811
Sortino Ratio                        -0.61812
Calmar Ratio                         -0.12252
Alpha [%]                           -57.05969
Beta                                  0.05939
Max. Drawdown [%]                   -69.93411
Avg. Drawdown [%]                    -9.57913
Max. Drawdown Duration     3279 days 00:00:00
Avg. Drawdown Duration      276 days 00:00:00
# Trades                          

In [31]:
bt = Backtest(new_ru, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True)
stats = bt.run()
bt.plot()
stats

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                    98.76645
Equity Final [$]                    102244.44
Equity Peak [$]                     198116.35
Commissions [$]                      57500.89
Return [%]                            2.24444
Buy & Hold Return [%]                10.37736
Return (Ann.) [%]                     0.23026
Volatility (Ann.) [%]                28.14006
CAGR [%]                              0.15324
Sharpe Ratio                          0.00818
Sortino Ratio                         0.01247
Calmar Ratio                          0.00406
Alpha [%]                             1.53465
Beta                                   0.0684
Max. Drawdown [%]                   -56.73828
Avg. Drawdown [%]                   -11.02295
Max. Drawdown Duration     2817 days 00:00:00
Avg. Drawdown Duration      240 days 00:00:00
# Trades                          

In [32]:
bt = Backtest(new_v, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True)
stats = bt.run()
bt.plot()
stats

Start                     2021-09-15 00:00:00
End                       2022-09-15 00:00:00
Duration                    365 days 00:00:00
Exposure Time [%]                    86.77686
Equity Final [$]                    85334.666
Equity Peak [$]                    112832.214
Commissions [$]                      4058.834
Return [%]                          -14.66533
Buy & Hold Return [%]               -34.52685
Return (Ann.) [%]                   -15.22273
Volatility (Ann.) [%]                22.83261
CAGR [%]                            -10.37105
Sharpe Ratio                         -0.66671
Sortino Ratio                        -0.83402
Calmar Ratio                         -0.62464
Alpha [%]                           -20.99791
Beta                                 -0.18341
Max. Drawdown [%]                    -24.3703
Avg. Drawdown [%]                    -7.61973
Max. Drawdown Duration      217 days 00:00:00
Avg. Drawdown Duration       54 days 00:00:00
# Trades                          